# **Q-TRANSFER LLC**

### **Short Summary**
This notebook uses a fictional knowledge base for **Q-Transfer**, a fictional fintech company, to build and test a **Retrieval-Augmented Generation (RAG)** application.
The main purpose of this notebook is to transform the knowledge base into numerical vector representations, store those vectors in a **vector database**, and visualise the resulting vector space in **2D and 3D**.

### **Notebook Structure**
This notebook is divided into three major parts:

**Part 1:** Converting the Input Documents into Chunks.

**Part 2:** Converting Chunks into Vectors and Storing Them in Chroma.

**Part 3:** Visualising the Vector Store in 2D and 3D Vector Space.

### **Installation**
The required Python libraries can be installed using `pip`. You can simply copy and paste in your teminal
```bash
pip install python-dotenv gradio openai langchain-text-splitters langchain-community langchain-huggingface langchain-chroma langchain-openai tiktoken scikit-learn plotly
```

### **RAG Workflow**
The overall workflow in this notebook is:
**Documents → Chunks → Embeddings → Vector Store → Similarity Search → Retrieved Context**
Each document is first split into smaller chunks. The chunks are then converted into vector representations using an embedding model and stored in Chroma. These vectors can later be used to retrieve relevant information based on a user's query.

### **Objectives**
By the end of this notebook, I aim to understand how to:
- Load and process documents from a knowledge base
- Split documents into meaningful chunks
- Generate embeddings for each chunk
- Store and manage embeddings using Chroma
- Perform similarity searches against the vector store
- Visualise high-dimensional embeddings in 2D and 3D
- Understand how chunk size and overlap affect retrieval

### **Embedding Model**
The text chunks are converted into vector representations using a Hugging Face embedding model and the OpenAI embedding model.
The resulting vectors represent the semantic meaning of the text and allow similar pieces of information to be identified through vector similarity.

### **Vector Visualisation**
The embeddings generated by the model exist in a high-dimensional vector space. Since these dimensions cannot be directly visualised, dimensionality reduction is used to project the vectors into 2D and 3D spaces.
This notebook uses **t-SNE** to create these lower-dimensional representations for visualisation.

### **Part 1: Converting the Input Documents into Chunks.**

In [1]:
# import the necessary libraries

import os
import glob
import tiktoken
import numpy as np
import plotly.graph_objects as go
import gradio as gr
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
from pathlib import Path
from openai import OpenAI

c:\Users\DELL\Documents\LLM Projects\q_transfer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\DELL\AppData\Local\Temp\ipykernel_16396\2272925242.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [2]:
# Constants

MODEL = "gpt-4.1-nano"
HF_DATABASE = "hf_vector_db"
OAI_DATABASE = "oai_vector_db"

In [3]:
# Load the OpenAI API KEY from .env

load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print("OpenAI Key successfully loaded.")
else:
    print("OpenAI key not loaded.")


OpenAI Key successfully loaded.


In [4]:
# Create an instance of the OpenAI python client library

openai = OpenAI()

In [5]:
# Display the number of files and characters in the knowledge base

knowledge_base_path = "../q_transfer_knowledge_base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True) # recursive=True tells Python to search inside subfolders too, not just the main folder.
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, "r", encoding="utf-8") as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in the knowledge base: {len(entire_knowledge_base):,}")

Found 39 files in the knowledge base
Total characters in the knowledge base: 24,269


In [6]:
# print the entire knowledge base
# print(entire_knowledge_base)

In [7]:
# Tokenize the entire knowledge base and get the number of tokens

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
print(f"The number of {MODEL} converted to is: {len(tokens):,}")

The number of gpt-4.1-nano converted to is: 4,267


In [8]:
# display the tokens and their representations

# for token in tokens:
#     print(token, repr(encoding.decode([token])))

In [9]:
# Load in everything in the knowledge base using Langchain's Loaders

folders = glob.glob("../q_transfer_knowledge_base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
    folder_docs = loader.load()
    # print(f"Folder Docs: {folder_docs}\n")
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents\n")
print(f"Documents: {documents[3]}")

Loaded 39 documents

Documents: page_content='# AML and CFT Controls

Anti-Money Laundering and Countering the Financing of Terrorism controls help Q-Transfer identify and manage financial crime risk.

## Key Controls

- Customer due diligence
- Transaction monitoring
- Sanctions screening
- Risk assessment
- Suspicious activity investigation
- Enhanced due diligence
- Regulatory reporting where required

A transaction being unusual does not automatically mean that it is illegal. Alerts require appropriate investigation according to company procedures.
' metadata={'source': '..\\q_transfer_knowledge_base\\compliance\\aml-cft.md', 'doc_type': 'compliance'}


In [10]:
# Divide the documents into chucks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

print(f"Document Token Size: {len(tokens):,}")
print(f"Divided into {len(chunks)} chunks")
# print(f"First chunk:\n\n{chunks}")

Document Token Size: 4,267
Divided into 73 chunks


In [11]:
# Display each chunk

# chunk_num = 1
# for chunk in chunks:
#     print(f"## CHUNK NUMBER {chunk_num}")
#     print(f"Chunk {chunk}")
#     chunk_num += 1
#     print("\n\n")

In [12]:
# Since there are 79 chunks let's see what's in chunk 50
chunks[50]

Document(metadata={'source': '..\\q_transfer_knowledge_base\\risk\\fraud-management.md', 'doc_type': 'risk'}, page_content='Customer safety and preservation of evidence are important during investigations.')

### **Part 2: Convert Chunks to Vectors and Store in Chroma**

 After splitting the documents in the knowledge base to chunks, we're going to convert them into vectors and store them in a Chroma vector store.

If you come across this notebook and need to run it, you'll have to create a hugging face account using this link https://huggingface.co/join. After creating your account, you have to create a secret key. The secret key is needed to gain access to variety of models, datasets, spaces e.t.c.

### **Sign in to Hugging Face and Creating API Token**
If you don't have a hugging face account create one using this link https://huggingface.co/join

If you already have a hugging face account, sign in using this account https://huggingface.co/login

After sign in, navigate to Settings > Access Token > Create new token > Click on the WRITE tab to give write permissions > Give your token a key > and finally, generate the token


**NB: Remember to copy your token before closing the pop up. Else, you'd have to create a new token again if you didn't save your token somewhere accessible or if misplaced. In addition, remember to store your token in a .env file. If your token or API key goes public, hugging face automatically render it inactive**

I'll be using Hugging Face Embedding and OpenAI Embedding to experiment on how different embeddings function

In [13]:
# Create 2 vector stores

hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(HF_DATABASE) and os.path.exists(OAI_DATABASE):
    Chroma(persist_directory=HF_DATABASE, embedding_function=hf_embeddings).delete_collection()
    Chroma(persist_directory=OAI_DATABASE, embedding_function=openai_embeddings).delete_collection()

hf_vector_store = Chroma.from_documents(documents=chunks, embedding=hf_embeddings, persist_directory=HF_DATABASE)
oai_vector_store = Chroma.from_documents(documents=chunks, embedding=openai_embeddings, persist_directory=OAI_DATABASE)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2805.39it/s]


In [14]:
# Display the number of documents stored in each vector store

print(f"There are {hf_vector_store._collection.count()} vectors in the Hugging Face vector store")
print(f"There are {oai_vector_store._collection.count()} vectors in the OpenAI vector store")

There are 73 vectors in the Hugging Face vector store
There are 73 vectors in the OpenAI vector store


In [15]:
# Function to display info about each vector store

def display_vector_dimension(vectorstore):
    collection = vectorstore._collection
    # print(f"Collection: {collection}")
    count = collection.count()

    sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
    dimensions = len(sample_embedding)
    print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

In [16]:
# Call the hugging face vector store
display_vector_dimension(hf_vector_store)

There are 73 vectors with 384 dimensions in the vector store


In [17]:
# Call the OpenAI vector store
display_vector_dimension(oai_vector_store)

There are 73 vectors with 3,072 dimensions in the vector store


### **Part 3: Visualizing the Vector Data Store**

I'll be displaying the vectors stored in the datastore in 2D and 3D

In [18]:
# Vector data store

def display_2D_of_the_vector_store(selected_vector_store):

    result = selected_vector_store._collection.get(
        include=["embeddings", "documents", "metadatas"]
    )

    vectors = np.array(result["embeddings"])
    documents = result["documents"]
    metadatas = result["metadatas"]

    doc_types = [metadata["doc_type"] for metadata in metadatas]

    # Get unique document types
    unique_types = list(set(doc_types))

    # Create a colour for each document type
    palette = [
        "blue", "green", "red", "orange", "purple",
        "pink", "brown", "gray", "cyan", "yellow"
    ]

    color_map = {
        doc_type: palette[i % len(palette)]
        for i, doc_type in enumerate(unique_types)
    }

    colors = [color_map[t] for t in doc_types]

    tsne = TSNE(n_components=2, random_state=42)

    reduced_vectors = tsne.fit_transform(vectors)

    fig = go.Figure(
        data=[
            go.Scatter(
                x=reduced_vectors[:, 0],
                y=reduced_vectors[:, 1],
                mode="markers",
                marker=dict(
                    size=5,
                    color=colors,
                    opacity=0.8
                ),
                text=[
                    f"Type: {t}<br>Text: {d[:100]}..."
                    for t, d in zip(doc_types, documents)
                ],
                hoverinfo="text"
            )
        ]
    )

    fig.update_layout(
        title="2D Chroma Vector Store Visualization",
        xaxis_title="x",
        yaxis_title="y",
        width=800,
        height=600,
        margin=dict(r=20, b=10, l=10, t=40)
    )

    fig.show()

In [19]:
# Display 2D representation of the HuggingFace vector

display_2D_of_the_vector_store(hf_vector_store)

In [20]:
# Display 2D representation of the OpenAI vector

display_2D_of_the_vector_store(oai_vector_store)

In [21]:
def display_3D_of_the_vector_store(selected_vector_store):

    result = selected_vector_store._collection.get(
        include=["embeddings", "documents", "metadatas"]
    )

    vectors = np.array(result["embeddings"])
    documents = result["documents"]
    metadatas = result["metadatas"]

    doc_types = [metadata["doc_type"] for metadata in metadatas]

    unique_types = list(set(doc_types))

    palette = [
        "blue", "green", "red", "orange", "purple",
        "pink", "brown", "gray", "cyan", "yellow"
    ]

    color_map = {
        doc_type: palette[i % len(palette)]
        for i, doc_type in enumerate(unique_types)
    }

    colors = [color_map[t] for t in doc_types]

    tsne = TSNE(n_components=3, random_state=42)
    reduced_vectors = tsne.fit_transform(vectors)

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=reduced_vectors[:, 0],
                y=reduced_vectors[:, 1],
                z=reduced_vectors[:, 2],
                mode="markers",
                marker=dict(
                    size=5,
                    color=colors,
                    opacity=0.8
                ),
                text=[
                    f"Type: {t}<br>Text: {d[:100]}..."
                    for t, d in zip(doc_types, documents)
                ],
                hoverinfo="text"
            )
        ]
    )

    fig.update_layout(
        title="3D Chroma Vector Store Visualization",
        scene=dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z"
        ),
        width=900,
        height=700,
        margin=dict(r=10, b=10, l=10, t=40)
    )

    fig.show()

In [22]:
# 3D representation of the HuggingFace vector store

display_3D_of_the_vector_store(hf_vector_store)

In [23]:
# 3D representation of the OpenAI vector store

display_3D_of_the_vector_store(oai_vector_store)